# 03 - Estatistica e testes estatisticos em RHEste notebook pratica estatistica com dados de RH. Vamos calcular medidas descritivas, correlacao, teste t e qui-quadrado.O objetivo nao e decorar formulas. O objetivo e aprender a fazer perguntas melhores e interpretar resultados com cuidado.

In [ ]:
# pathlib localiza arquivos.from pathlib import Path# pandas trabalha com tabelas.import pandas as pd# scipy.stats contem funcoes estatisticas prontas.from scipy import stats# Localizamos a pasta de dados.BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()DADOS = BASE_DIR / "dados"# Abrimos as bases usadas neste notebook.colaboradores = pd.read_csv(DADOS / "colaboradores.csv")treinamentos = pd.read_csv(DADOS / "treinamentos.csv")

## 1. Media, mediana e dispersaoMedia mostra um centro aritmetico. Mediana mostra o valor central. Desvio padrao mostra espalhamento. Em salarios, olhar apenas media pode enganar.

In [ ]:
# Agrupamos por nivel porque comparar salarios sem considerar nivel seria injusto.resumo_salario = (    colaboradores.groupby("nivel")["salario_mensal"]    .agg(        qtd="count",        media="mean",        mediana="median",        desvio="std",        minimo="min",        maximo="max",    )    .round(1))resumo_salario

## 2. CorrelacaoCorrelacao mede associacao entre variaveis numericas. Ela nao prova causa. Aqui vamos observar se absenteismo, horas extras e divergencias de ponto caminham juntos.

In [ ]:
# Selecionamos apenas colunas numericas relevantes para a pergunta.colunas_correlacao = [    "absenteismo_dias_ano",    "horas_extras_mes",    "divergencias_ponto_mes",    "nota_performance",]# corr() calcula correlacao entre cada par de colunas.correlacao = colaboradores[colunas_correlacao].corr()# Arredondamos para facilitar leitura.correlacao.round(2)

## 3. Teste t: comparacao de mediasPergunta: treinamentos presenciais e online tem notas medias de eficacia diferentes?Hipotese nula: as medias sao iguais.Hipotese alternativa: as medias sao diferentes.

In [ ]:
# Algumas notas podem estar vazias porque nem todo mundo respondeu.# dropna remove linhas sem nota para este teste especifico.notas = treinamentos.dropna(subset=["nota_eficacia"]).copy()# Separamos as notas em dois grupos.presencial = notas.loc[notas["modalidade"] == "Presencial", "nota_eficacia"]online = notas.loc[notas["modalidade"] == "Online", "nota_eficacia"]# ttest_ind compara medias de dois grupos independentes.# equal_var=False usa a versao de Welch, mais conservadora quando variancias podem diferir.resultado_t = stats.ttest_ind(presencial, online, equal_var=False)print(f"Quantidade presencial: {len(presencial)}")print(f"Quantidade online: {len(online)}")print(f"Media presencial: {presencial.mean():.2f}")print(f"Media online: {online.mean():.2f}")print(f"p-valor: {resultado_t.pvalue:.4f}")

### Como interpretar o p-valorSe o p-valor for menor que 0,05, costumamos dizer que ha evidencia estatistica contra a hipotese nula. Se for maior ou igual a 0,05, nao temos evidencia suficiente para afirmar diferenca estatistica.Isso nao mede importancia pratica. Tambem nao prova causalidade.

## 4. Qui-quadrado: associacao entre categoriasPergunta: existe associacao entre treinamento obrigatorio e conclusao?Aqui as duas variaveis sao categoricas: `obrigatorio` e `concluiu`.

In [ ]:
# crosstab cria uma tabela de contingencia.tabela = pd.crosstab(treinamentos["obrigatorio"], treinamentos["concluiu"])# chi2_contingency aplica o teste qui-quadrado.chi2, p_valor, graus_liberdade, esperados = stats.chi2_contingency(tabela)# display mostra a tabela no notebook.display(tabela)print(f"Estatistica qui-quadrado: {chi2:.3f}")print(f"Graus de liberdade: {graus_liberdade}")print(f"p-valor: {p_valor:.4f}")

## 5. Exercicios1. Compare salario medio e mediano entre dois niveis.2. Calcule a correlacao entre `horas_treinamento_ano` e `nota_performance`.3. Crie uma hipotese sobre horas extras e teste com media por area.4. Escreva uma conclusao usando a frase: `neste conjunto de dados didatico...`.5. Escreva uma limitacao etica ou metodologica da sua analise.